# Building AI Coding Agents with OpenClaw and MiniMax

A practical introduction to **tool-calling agents** and **self-improving research loops**, built around a single working example: an agent that improves a chess bot by editing its source code, with an automatic evaluator scoring every attempt.

## What you will be able to do after working through this notebook

1. Describe the **four-layer pattern** that structures a tool-calling agent — *Gateway, Context, ReAct, Tools* — and locate the file in this repository where each layer lives.
2. Read an **agent trace** and explain what happened on each round.
3. **Modify a tool's description** and predict how that change reaches the model.
4. Run a **multi-iteration research loop** and interpret the resulting evaluation curve.
5. Distinguish what an **eval-driven loop** catches that a single prompt cannot — and identify where its limits begin.

## How to work through it

Run the cells in order. Each section opens with a short conceptual introduction and ends with a brief reflection prompt. The code cells are the actual lab work; the markdown around them is the explanation.

This notebook supports two modes:

- **Mock mode** (default, no API key required) — the model's responses are deterministic and scripted. Good for learning the *mechanics* of the loop: the schemas, the trace structure, the accept/reject step.
- **Live mode** — set `MINIMAX_API_KEY` in Colab Secrets. The model's behaviour becomes nondeterministic, which is when the *interesting* properties of an autoresearch loop appear: real variance in patches, real differences in trajectory.

Both modes teach. Some lessons (notably the lesson in *"Tools are descriptions the model reads"* about how tool descriptions shape behaviour) only become visible in live mode; the notebook flags those explicitly.

Start with the bootstrap cell below.

In [ ]:
"""Colab bootstrap — clone the repo, install deps, wire up MiniMax key,
snapshot the pristine bot files so each lab step can reset cleanly.

Idempotent — safe to re-run.
"""
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nickita-khylkouski/autoresearch-brief-challenge"
REPO_DIR = "autoresearch-brief-challenge"
# TODO: drop `-b feat/3-tool-calling-openclaw-agent` once that branch merges
# to main. Until then, the OpenClaw four-layer agent code lives only on
# that branch.
CLONE_BRANCH = "feat/3-tool-calling-openclaw-agent"

def _is_repo_root(p: Path) -> bool:
    return (p / "pyproject.toml").exists() and (p / "autoresearch_chess").is_dir()

# Idempotent: if we're already inside the repo (e.g. this cell was re-run),
# don't clone again — that would create nested autoresearch-brief-challenge/
# autoresearch-brief-challenge/... directories. Only clone if no repo root is
# reachable from cwd or its parents.
cwd = Path.cwd().resolve()
if _is_repo_root(cwd):
    pass  # already at the repo root
elif _is_repo_root(cwd / REPO_DIR):
    os.chdir(cwd / REPO_DIR)
else:
    # Walk up to find an existing checkout before falling back to cloning.
    found = next((p for p in cwd.parents if _is_repo_root(p)), None)
    if found is not None:
        os.chdir(found)
    elif IN_COLAB:
        print(f"Cloning {REPO_URL} (branch: {CLONE_BRANCH}) ...")
        subprocess.run(["git", "clone", "--quiet", "-b", CLONE_BRANCH, REPO_URL], check=True)
        os.chdir(REPO_DIR)

print(f"Working directory: {Path.cwd()}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-chess"], check=True)
print("python-chess installed.")

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Force all four editable bot files to their pristine baseline. The HEAD-of-
# branch versions of evaluate.py and search.py have improvements from past
# AutoResearch runs (passed-pawn evaluation, quiescence search). Resetting all
# four ensures the loop always starts from the documented ~630 Elo baseline.
PRISTINE_BOT_FILES = {
    "config.py": """SEARCH_DEPTH = 1
MATERIAL_WEIGHT = 1.0
MOBILITY_WEIGHT = 0.0
KING_SAFETY_WEIGHT = 0.0
USE_PIECE_SQUARES = False
CAPTURE_FIRST = True
""",
    "evaluate.py": """from __future__ import annotations

import chess

from . import config


PIECE_VALUES = {
    chess.PAWN: 100,
    chess.KNIGHT: 320,
    chess.BISHOP: 330,
    chess.ROOK: 500,
    chess.QUEEN: 900,
    chess.KING: 0,
}

CENTER_SQUARES = {chess.D4, chess.E4, chess.D5, chess.E5}


def material_score(board: chess.Board) -> int:
    score = 0
    for piece_type, value in PIECE_VALUES.items():
        score += len(board.pieces(piece_type, chess.WHITE)) * value
        score -= len(board.pieces(piece_type, chess.BLACK)) * value
    return score


def piece_square_score(board: chess.Board) -> int:
    score = 0
    for square, piece in board.piece_map().items():
        file_idx = chess.square_file(square)
        rank_idx = chess.square_rank(square)
        center_bonus = 14 - (abs(file_idx - 3.5) + abs(rank_idx - 3.5)) * 4
        if square in CENTER_SQUARES and piece.piece_type in {chess.PAWN, chess.KNIGHT, chess.BISHOP}:
            center_bonus += 12
        score += int(center_bonus) if piece.color == chess.WHITE else -int(center_bonus)
    return score


def mobility_score(board: chess.Board) -> int:
    if board.is_game_over():
        return 0
    turn = board.turn
    board.turn = chess.WHITE
    white_moves = board.legal_moves.count()
    board.turn = chess.BLACK
    black_moves = board.legal_moves.count()
    board.turn = turn
    return white_moves - black_moves


def king_safety_score(board: chess.Board) -> int:
    score = 0
    for color, sign in ((chess.WHITE, 1), (chess.BLACK, -1)):
        king_square = board.king(color)
        if king_square is None:
            continue
        attackers = board.attackers(not color, king_square)
        defenders = board.attackers(color, king_square)
        score += sign * (len(defenders) * 8 - len(attackers) * 20)
    return score


def evaluate(board: chess.Board) -> int:
    if board.is_checkmate():
        return -100_000 if board.turn == chess.WHITE else 100_000
    if board.is_stalemate() or board.is_insufficient_material():
        return 0

    score = int(config.MATERIAL_WEIGHT * material_score(board))
    if config.USE_PIECE_SQUARES:
        score += piece_square_score(board)
    score += int(config.MOBILITY_WEIGHT * mobility_score(board))
    score += int(config.KING_SAFETY_WEIGHT * king_safety_score(board))
    return score
""",
    "move_ordering.py": """from __future__ import annotations

import chess

from . import config
from .evaluate import PIECE_VALUES


def move_score(board: chess.Board, move: chess.Move) -> int:
    score = 0
    if board.is_capture(move):
        victim = board.piece_at(move.to_square)
        attacker = board.piece_at(move.from_square)
        if victim:
            score += PIECE_VALUES.get(victim.piece_type, 0)
        if attacker:
            score -= PIECE_VALUES.get(attacker.piece_type, 0) // 10
        score += 1_000
    if move.promotion:
        score += PIECE_VALUES.get(move.promotion, 0)
    if board.gives_check(move):
        score += 75
    if not config.CAPTURE_FIRST:
        score = -score
    return score


def ordered_moves(board: chess.Board) -> list[chess.Move]:
    moves = list(board.legal_moves)
    return sorted(moves, key=lambda move: move_score(board, move), reverse=True)
""",
    "search.py": """from __future__ import annotations

import chess

from . import config
from .evaluate import evaluate
from .move_ordering import ordered_moves


def _search(board: chess.Board, depth: int, alpha: int, beta: int) -> int:
    if depth <= 0 or board.is_game_over():
        return evaluate(board)

    if board.turn == chess.WHITE:
        value = -10**9
        for move in ordered_moves(board):
            board.push(move)
            value = max(value, _search(board, depth - 1, alpha, beta))
            board.pop()
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return value

    value = 10**9
    for move in ordered_moves(board):
        board.push(move)
        value = min(value, _search(board, depth - 1, alpha, beta))
        board.pop()
        beta = min(beta, value)
        if alpha >= beta:
            break
    return value


def choose_move(board: chess.Board, depth_limit: int | None = None) -> chess.Move:
    moves = ordered_moves(board)
    if not moves:
        raise ValueError("No legal moves")
    depth = max(1, min(depth_limit or config.SEARCH_DEPTH, config.SEARCH_DEPTH))

    best_move = moves[0]
    if board.turn == chess.WHITE:
        best_score = -10**9
        for move in moves:
            board.push(move)
            score = _search(board, depth - 1, -10**9, 10**9)
            board.pop()
            if score > best_score:
                best_score = score
                best_move = move
    else:
        best_score = 10**9
        for move in moves:
            board.push(move)
            score = _search(board, depth - 1, -10**9, 10**9)
            board.pop()
            if score < best_score:
                best_score = score
                best_move = move
    return best_move
""",
}

for name, content in PRISTINE_BOT_FILES.items():
    Path(f"bot/{name}").write_text(content, encoding="utf-8")

def reset_bot_to_baseline():
    """Restore all bot/*.py files to their pristine snapshot."""
    for name, content in PRISTINE_BOT_FILES.items():
        Path(f"bot/{name}").write_text(content, encoding="utf-8")

print(f"Forced {len(PRISTINE_BOT_FILES)} bot/*.py files to pristine baseline (~630 Elo).")

# Wire up the MiniMax key from Colab Secrets (if any); otherwise force mock.
if IN_COLAB:
    try:
        from google.colab import userdata
        key = userdata.get("MINIMAX_API_KEY")
    except Exception:
        key = None
    if key:
        os.environ["MINIMAX_API_KEY"] = key
        os.environ.pop("AUTORESEARCH_MOCK_MINIMAX", None)
        print("MINIMAX_API_KEY loaded from Colab Secrets — live calls available.")
    else:
        os.environ["AUTORESEARCH_MOCK_MINIMAX"] = "1"
        print("Mock mode (no MINIMAX_API_KEY in Colab Secrets — full lab still works).")
elif not os.environ.get("MINIMAX_API_KEY"):
    os.environ["AUTORESEARCH_MOCK_MINIMAX"] = "1"
    print("Local mode without MINIMAX_API_KEY — mock mode.")

def use_mock():
    """Read the current mock-mode setting (toggled by the bootstrap above)."""
    return os.environ.get("AUTORESEARCH_MOCK_MINIMAX") == "1"

print(f"Python: {sys.version.split()[0]}")
print(f"Mock mode: {use_mock()}")

## 1 · The outcome you'll produce

Before any architecture, here is what the work in this notebook produces: an agent that climbs a chess bot's estimated Elo from roughly 630 to roughly 1280 by proposing a small number of code patches — each patch only accepted if a fixed evaluator measures an actual improvement.

The image below is from a captured run. By the end of this notebook you will have produced a curve like this from your own machine.

**Why show the outcome first.** The architecture and tool-calling mechanics that follow are means to this end, not ends in themselves. Anchoring the goal makes the rest of the explanation easier to follow.

In [ ]:
"""§0 — Display a captured Elo curve. This is your target."""
from pathlib import Path
from IPython.display import Image, display

replay_png = Path("artifacts/demo_replay/progress.png")
if replay_png.exists():
    display(Image(str(replay_png)))
else:
    print("(Captured progress.png not present — your own curve appears in §4.)")

## 2 · How a tool-calling agent is structured

A tool-calling agent is not one piece of software. It is four cooperating components, each with a single responsibility. Together they form a pattern — *OpenClaw* — that you will see reproduced across many production agent frameworks once you know what to look for.

| Layer | File in `autoresearch_chess/agent/` | Responsibility |
|---|---|---|
| **Gateway** | `gateway.py` | Entry point. Receives a request, returns a result. |
| **Context** | `context.py` | Assembles the conversation the model sees: system prompt, prior turns, current task. |
| **ReAct** | `react.py` | Drives the model–tool–model–tool cycle until a stop condition is reached. |
| **Tools** | `tools.py` | The five functions the model is permitted to call. |

The model itself (MiniMax) sits outside these four layers — the agent calls the model the way an application calls a database. The Gateway layer is the seam that makes the agent **swappable across deployments**: in-process today inside this notebook, behind a service boundary tomorrow.

Run the next cell to see all four layers as they actually exist in this repository.

In [ ]:
"""§1 — Architecture in code. Each OpenClaw layer at a glance."""
import inspect
from autoresearch_chess.agent.gateway import ChessAgent
from autoresearch_chess.agent.context import build_initial_conversation
from autoresearch_chess.agent.react import run_react_loop
from autoresearch_chess.agent.tools import TOOLS

print("=== Tool layer — 5 tools ===")
for t in TOOLS:
    marker = "🛑 terminal" if t.terminal else "📖 read-only"
    print(f"  {marker}  {t.name}")
    print(f"             {t.description}")

print()
print("=== Context layer ===")
print(f"  {inspect.signature(build_initial_conversation)}")

print()
print("=== ReAct layer ===")
print(f"  {inspect.signature(run_react_loop)}")

print()
print("=== Gateway layer ===")
print(f"  class ChessAgent — entry point that wires the three above together")

print()
print("To see a full layer's source, run:")
print("  print(inspect.getsource(ChessAgent))")
print("Or open the file from Colab's left sidebar (folder icon).")

## 3 · One iteration, end to end

Before running many iterations, run *one* — and read carefully what it produced.

The next cell drives a single complete iteration of the loop:

1. The agent is told the baseline Elo and the list of editable files.
2. It calls tools to inspect the bot's source — `list_bot_files`, `read_bot_file`.
3. It calls the terminal `edit_file(path, old_str, new_str)` tool, which synthesizes a unified diff under the hood.
4. The evaluator runs the patched bot in a sandbox and produces a score.
5. The loop **accepts** the patch if the score improved by at least the threshold, otherwise it **rejects**.

Three points worth noticing when you read the code:

- **The agent never edits the filesystem directly.** Its only mutating tool is `edit_file`, and the synthesized diff passes through a guardrail check before the evaluator sees it. Constraint enforcement is at the tool layer, not in the prompt.
- **The same code runs in mock and live mode.** The only difference is which `chat_with_tools` function gets called inside `react.py` — the agent itself does not know.
- **The iteration produces a durable record** (`agent_trace.jsonl`, `decision.json`, `patch.diff`). Agents are easier to debug — and easier to study — when their reasoning is written down.

This takes 30–60 seconds in mock mode.

In [ ]:
"""Lab Step 1-3 — Run one full agent iteration end-to-end."""
from dataclasses import replace
from autoresearch_chess.loop import run_loop
from autoresearch_chess.config import STAGE_LOOP_CONFIG

reset_bot_to_baseline()
config = replace(STAGE_LOOP_CONFIG, iterations=1, mock_minimax=use_mock())
print(f"Running one iteration (mock={use_mock()})...")
print()
summary_baseline = run_loop(config)

# summary_baseline['accepted'] is a COUNT of accepted iterations (0 or 1 here),
# not a boolean — be explicit so this stays correct if iterations changes.
accepted_this_run = summary_baseline["accepted"] > 0

print()
print(f"=== ITERATION COMPLETE ===")
print(f"Run dir:        {summary_baseline['run_dir']}")
print(f"Decision:       {'✅ ACCEPTED' if accepted_this_run else '❌ rejected'}")
print(f"Baseline Elo:   {summary_baseline['baseline_eval'].get('estimated_elo'):.1f}")
print(f"Best Elo:       {summary_baseline['best_eval'].get('estimated_elo'):.1f}")

In [ ]:
"""Lab Step 1-3 — Read your agent's tool-call trace.

`agent_trace.jsonl` has one event per ReAct round. This is the agent's
reasoning, made inspectable.

If the agent iteration errored out before a single ReAct round completed
(e.g. live MiniMax auth/network failure), the trace file won't exist.
We surface the error from decision.json instead so you can diagnose.
"""
import json
from pathlib import Path

run_dir = Path(summary_baseline["run_dir"])
iter_dir = run_dir / "iterations/001"
trace_path = iter_dir / "agent_trace.jsonl"

if trace_path.exists():
    for line in trace_path.read_text(encoding="utf-8").splitlines():
        event = json.loads(line)
        calls = ", ".join(tc["name"] for tc in event["tool_calls"]) or "<final message>"
        print()
        print(f"round {event['round']}: {calls}")
        for tc in event["tool_calls"]:
            preview = tc["result_preview"].replace("\n", " ")[:140]
            print(f"  args:    {tc['arguments']}")
            print(f"  result:  {preview}...")
    print()
    print()
    print(f"Full trace on disk: {trace_path}")
    print("Open it via Colab's file browser (folder icon, left sidebar).")
else:
    # No trace = the iteration errored before any ReAct round completed.
    print("⚠️  No agent_trace.jsonl was written — the iteration failed before any")
    print("ReAct round completed. The reason is in decision.json:")
    print()
    decision_path = iter_dir / "decision.json"
    if decision_path.exists():
        decision = json.loads(decision_path.read_text(encoding="utf-8"))
        for reason in decision.get("reasons", []):
            print(f"  • {reason}")
        print()
        if not use_mock():
            print("You're in LIVE mode. Most common causes for a first-call failure:")
            print("  - MINIMAX_API_KEY invalid or expired")
            print("  - MiniMax API host unreachable from Colab")
            print("  - Wrong API mode (try setting MINIMAX_API_MODE=openai in Colab Secrets)")
            print()
            print("Quickest workaround: clear MINIMAX_API_KEY from Colab Secrets, re-run")
            print("the bootstrap cell (mock mode kicks in), then re-run Lab Step 1-3.")
        else:
            print("You're in mock mode — this shouldn't normally happen. Re-run bootstrap.")
    else:
        print(f"  (decision.json missing — check {iter_dir})")

### Concept check

Confirm before moving on that you can answer these:

1. **Three rounds.** What did the agent do on rounds 1, 2, and 3, and why does the third round end the iteration?
2. **One terminal tool.** `edit_file` is the only terminal tool. Why is that important — what failure mode does it prevent?
3. **The trace persists.** Where on disk does the trace live, and what would you use it for?

If anything is unclear, scroll up and re-read the trace output before continuing. The next sections build on these mechanics.

## 4 · Tools are descriptions the model reads

A useful idea, and one that surprises most people the first time they see it: **tool descriptions are not internal documentation. They are part of the model's input.**

From the model's perspective, each tool is a JSON object — name, description, and schema — concatenated into the same context window as the conversation. Rewriting the description changes what the model reads, which can change what it does. The string you write in `description=` is just as much "the prompt" as the system message is.

The next cell makes this visible without editing any files. Pick a tool, replace its description with something different, and inspect the before/after JSON that MiniMax actually receives.

- **In live mode**, the new description may cause the model to choose tools differently — for instance, a vague description on `edit_file` can lead the model to call it earlier or not at all.
- **In mock mode**, the model's choices are scripted and won't change — but the schema diff is still real. Seeing it is itself the lesson: **tool descriptions are how you steer a tool-calling model**.

In [ ]:
"""Lab Step 4 — Modify one tool's description in memory.

Pick a tool from the dropdown and type a new description. The change applies
to TOOLS in this notebook session only (restart kernel to undo).
"""
import json
from dataclasses import replace as dc_replace
from autoresearch_chess.agent import tools as tools_module

tool_to_modify = "edit_file"  # @param ["list_bot_files", "read_bot_file", "get_baseline_eval", "get_recent_history", "edit_file"]
new_description = "Does something to a file. Use this when you need to."  # @param {type:"string"}

original = next(t for t in tools_module.TOOLS if t.name == tool_to_modify)
modified = dc_replace(original, description=new_description)
idx = tools_module.TOOLS.index(original)
tools_module.TOOLS[idx] = modified
tools_module.TOOLS_BY_NAME[tool_to_modify] = modified

print(f"=== BEFORE ===")
print(json.dumps(original.to_openai_format(), indent=2))
print()
print(f"=== AFTER ===")
print(json.dumps(modified.to_openai_format(), indent=2))
print()
print("The second JSON above is what MiniMax now receives as the tool definition.")

In [ ]:
"""Lab Step 4 — Re-run an iteration with the modified tool.

In LIVE mode, the model will read your new description and may behave differently.
In MOCK mode, the trace is scripted — so you won't see a behavior difference,
but you can verify the modified schema was sent.
"""
from dataclasses import replace
from autoresearch_chess.loop import run_loop
from autoresearch_chess.config import STAGE_LOOP_CONFIG

reset_bot_to_baseline()
config = replace(STAGE_LOOP_CONFIG, iterations=1, mock_minimax=use_mock())
summary_modified = run_loop(config)

print()
print(f"Run dir: {summary_modified['run_dir']}")
if use_mock():
    print("(Mock mode — trace is scripted; the schema change above is the result.)")
else:
    print("(Live mode — re-read iterations/001/agent_trace.jsonl above to spot any behavior change.)")

### Reflection

Two prompts before moving on:

1. **Compare the before and after JSON.** Exactly what changed, and what stayed the same?
2. **Predict.** If you had live access, which tool's description would you rewrite to most change the agent's behaviour, and why? Write a single sentence — testable predictions are the start of useful experiments.

## 5 · The loop — many small evaluations, not one large prompt

You have seen one iteration and one tool modification. The remaining piece is the *loop*: running the agent repeatedly, scoring each attempt, and keeping only the patches that improve a measurable objective.

This is the **AutoResearch** pattern in four parts:

- an **agent** that can read code, propose changes, and inspect recent history;
- an **objective evaluator** that produces a single number per attempt;
- a **bounded edit surface** so the agent cannot drift into unrelated code;
- an **accept-or-reject step** that compares each new score against the current best.

**Why this is qualitatively different from one prompt.** A single prompt has to be right on the first try. A loop is allowed to be wrong many times; the evaluator's job is to catch the wrong attempts and keep the right ones. The whole pattern is the difference between a chatbot answering once and an agent **converging on a measurable outcome**.

The same shape appears at much larger scale. MiniMax reports running an analogous loop on M2.7 during training: the model proposes changes to its own scaffolding, an internal eval scores each attempt, and only the improvements are kept. Different domain, different magnitude, same pattern.

The next cell runs five iterations. Takes roughly three to five minutes.

In [ ]:
"""Lab Step 5 — Your AutoResearch loop. ~3-5 minutes."""
iterations = 5  # @param {type:"slider", min:1, max:10, step:1}

from dataclasses import replace
from autoresearch_chess.loop import run_loop
from autoresearch_chess.config import STAGE_LOOP_CONFIG

reset_bot_to_baseline()
print(f"Resetting bot to pristine baseline; running {iterations} iterations (mock={use_mock()})...")
print()

config = replace(STAGE_LOOP_CONFIG, iterations=iterations, mock_minimax=use_mock())
summary = run_loop(config)

print()
print(f"=== YOUR RUN ===")
print(f"Run ID:        {summary['run_id']}")
print(f"Baseline Elo:  {summary['baseline_eval'].get('estimated_elo'):.1f}")
print(f"Best Elo:      {summary['best_eval'].get('estimated_elo'):.1f}")
print(f"Accepted:      {summary['accepted']} / {summary['accepted'] + summary['rejected']}")
print(f"Run dir:       {summary['run_dir']}")

In [ ]:
"""Your Elo curve — the headline visual."""
from pathlib import Path
from IPython.display import Image, display

run_dir = Path(summary["run_dir"])
display(Image(str(run_dir / "progress.png")))

In [ ]:
"""Lab Step 5 — Inspect one accepted and one rejected patch."""
import json
from pathlib import Path

run_dir = Path(summary["run_dir"])
events = [
    json.loads(line)
    for line in (run_dir / "progress.jsonl").read_text(encoding="utf-8").splitlines()
]
iter_events = [e for e in events if e.get("event") == "iteration"]
first_accept = next((e for e in iter_events if e["decision"] == "accepted"), None)
first_reject = next((e for e in iter_events if e["decision"] == "rejected"), None)

def show(label, event):
    if event is None:
        print(f"=== {label}: none in this run ===\n")
        return
    n = event["iteration"]
    it_dir = run_dir / "iterations" / f"{n:03d}"
    decision = json.loads((it_dir / "decision.json").read_text(encoding="utf-8"))
    print(f"=== {label} · iteration {n} ===")
    print(f"candidate Elo: {event['candidate_elo']}    best Elo: {event['best_elo']}")
    print(f"reasons:       {decision.get('reasons', [])}")
    print(f"improvement:   {decision.get('improvement', 'n/a')}")
    print(f"--- patch ({it_dir / 'patch.diff'}) ---")
    diff_text = (it_dir / "patch.diff").read_text(encoding="utf-8")
    print("\n".join(diff_text.splitlines()[:30]))
    print()

show("ACCEPTED", first_accept)
show("REJECTED", first_reject)

### Reflection — read your curve

Spend a moment with the curve you just produced.

- **In mock mode**, you should see a sharp jump in iteration 1 (roughly 660 → 1280) followed by a flat plateau. **The plateau is not a failure.** It is the evaluator doing its job: once the best is already high, subsequent scripted patches no longer clear the accept threshold, so the loop correctly rejects them. *No patch is kept unless it improves the score.* That is the entire purpose of accept/reject.
- **In live mode**, your curve will differ from your neighbour's, because the model's choices are nondeterministic. Different patches, different curves, different best Elos. That variance is the loop expressing real choice under real evaluation.

Open `iterations/00X/patch.diff` for any accepted iteration. The patches are short — typically fewer than ten lines. **Compounding many small accepted patches is what produces the climb.** That is the pattern in one sentence.

### Concept check

1. **What would happen** if the accept threshold were set to zero? What about a very large positive number?
2. **What property of the evaluator** makes the accept/reject step trustworthy? What would break if that property were violated?

## 6 · Extensions — change one piece, observe the effect

You now have a working agent and a working loop. The fastest way to deepen understanding is to change exactly one piece of the system and observe the result. Three options, in increasing depth — pick one and try it.

- **Light touch — the system prompt.** Change what the agent is told at the start of each iteration.
- **Medium touch — add a tool.** Give the agent a new capability and re-run.
- **Deeper touch — bias the heuristic.** Inject domain-specific guidance into every iteration's kickoff message and compare the trajectory against your earlier run.

**A caveat that applies to all three.** In mock mode, the model's response is scripted and will not change in response to your modification. The modification is still real — the schema, the prompt, the tool list all change — and the change is visible if you switch to live mode. Mock mode shows you *what got sent to the model*; live mode shows you *what the model did with it*. Both are useful; they answer different questions.

### Light touch — modify the system prompt

The system prompt is where the agent is told its job and its constraints. Edit it. For example, ask the agent to write a short plain-English explanation of each patch before submitting. The placeholders `{editable}`, `{forbidden_files}`, `{baseline_elo}`, and `{best_elo}` are filled in at runtime — keep any you want substituted, drop the rest.

After running the cell below, scroll back up to *"The loop"* and re-run the five-iteration cell to see the new prompt take effect (visible in live mode).

In [ ]:
"""🟢 Beginner — Edit the agent's system prompt.

Edit the triple-quoted string below directly. (No Colab form field here:
Colab form annotations only work on single-line string assignments, and
a multi-line system prompt is more readable than escaping it onto one line.)
"""
custom_system_prompt = """You are MiniMax operating inside an AutoResearch chess loop.

Your job: improve the chess bot's estimated Elo by making one focused find/replace edit via edit_file(path, old_str, new_str).

BEFORE you call edit_file, write 2-3 sentences explaining in plain English what
heuristic your edit encodes and why you think it will help.

You may edit only: {editable}.
Do not edit: {forbidden_files}.

Baseline Elo: {baseline_elo}. Current best: {best_elo}.
Only patches that improve estimated Elo are accepted.

Use list_bot_files, read_bot_file, and get_recent_history to inspect the code,
then call edit_file(path, old_str, new_str) with a small targeted replacement."""

from autoresearch_chess.agent import context as ctx_module
ctx_module.SYSTEM_PROMPT = custom_system_prompt
print("Custom prompt installed. Scroll up to Lab Step 5 and re-run the 5-iteration cell.")
print()
print("Note: in mock mode, the trace stays scripted regardless of the prompt — switch")
print("to live mode (MINIMAX_API_KEY in Colab Secrets) to see your prompt actually take effect.")

### Medium touch — add a new tool

The agent currently has five tools. Add a sixth that lets it list files anywhere in the repository, not only the editable bot files. This widens the agent's *read-only* visibility without widening its *edit* surface: `edit_file` still rejects edits outside `EDITABLE_FILES`. Separating "what the model can see" from "what the model can change" is a useful design idea on its own.

The skeleton below shows the full `Tool` dataclass — name, description, JSON schema, handler. Adding a tool is one append to `TOOLS`. After the cell runs, the new tool will appear in `openai_tool_payload()` and be visible to the model on the next iteration (visible in live mode).

Once it works, try modifying the handler to filter by file extension, or write a different tool entirely — for example `grep_bot_files(pattern)`.

In [ ]:
"""🟡 Intermediate — Add a new `list_files` tool to the agent's toolkit."""
import json
from autoresearch_chess.agent.tools import Tool, TOOLS, TOOLS_BY_NAME

def list_files_handler(args, ctx):
    """Return entries (files and subdirs) under a relative path. Read-only."""
    path = str(args.get("path", ".")).strip() or "."
    target = ctx.root / path
    if not target.exists() or not target.is_dir():
        return json.dumps({"error": f"not_a_directory:{path}"})
    # Skip hidden dirs and noisy build artifacts
    entries = sorted(
        p.name + ("/" if p.is_dir() else "")
        for p in target.iterdir()
        if not p.name.startswith(".") and p.name not in ("__pycache__", "node_modules")
    )
    # Cap the listing so the agent isn't overwhelmed
    return json.dumps({"path": path, "entries": entries[:50]})

new_tool = Tool(
    name="list_files",
    description=(
        "List the files and subdirectories under a path relative to the repo root. "
        "Read-only. Use this to discover what code exists; you can still only edit "
        "the files returned by list_bot_files."
    ),
    schema={
        "type": "object",
        "properties": {
            "path": {
                "type": "string",
                "description": "Relative path from the repo root. Default: '.' (repo root).",
            }
        },
        "additionalProperties": False,
    },
    handler=list_files_handler,
)

if not any(t.name == new_tool.name for t in TOOLS):
    TOOLS.append(new_tool)
    TOOLS_BY_NAME[new_tool.name] = new_tool
    print(f"✅ Added '{new_tool.name}'. The agent now has {len(TOOLS)} tools.")
else:
    print(f"'{new_tool.name}' already added. {len(TOOLS)} tools registered.")

print()
print(f"Verify the new tool is in the schema MiniMax sees:")
from autoresearch_chess.agent.tools import openai_tool_payload
print(f"  Tool names: {[t['function']['name'] for t in openai_tool_payload()]}")

print()
print("In LIVE mode (MINIMAX_API_KEY set), re-run Lab Step 5 — the agent will see")
print("the new tool in its schema and may decide to call it. In mock mode, the agent")
print("follows its scripted flow regardless.")

### Deeper touch — bias the agent's heuristic

A real autoresearch loop usually needs to be **steered**: *"focus on endgame heuristics, not opening play"*, *"prefer move-ordering changes over evaluation changes"*, and so on. The lever for this is the user message that starts each iteration.

The cell below intercepts `build_initial_conversation` — the Context layer — so every iteration appends your chosen heuristic guidance to the kickoff message. It then runs five iterations and compares the resulting best Elo against your earlier run.

In live mode, different biases produce different trajectories — try several and watch the curves diverge. In mock mode the trajectory is fixed regardless of bias (the patches are scripted): the bias text reaches the model, but the scripted mock ignores it.

In [ ]:
"""🔴 Advanced — Bias the agent toward a chess heuristic and re-run.

Pick a heuristic bias in the form below. The cell monkey-patches the gateway's
view of `build_initial_conversation` so every iteration of the loop sees your
guidance appended to the kickoff user message. Then runs 5 iterations and
compares the resulting best Elo against your §4 baseline (in `summary`).
"""
heuristic_bias = "Focus your patches on endgame heuristics: king activity in the endgame, passed-pawn evaluation, and opposition. Avoid changes that primarily help opening play."  # @param {type:"string"}

from dataclasses import replace
from autoresearch_chess.loop import run_loop
from autoresearch_chess.config import STAGE_LOOP_CONFIG
from autoresearch_chess.agent import context as ctx_module
from autoresearch_chess.agent import gateway as gw_module

# Wrap the real build_initial_conversation so the bias lands in the user message
# of every iteration. The gateway looks up `build_initial_conversation` in its
# own module namespace at call time, so patching `gw_module` is what matters.
_original_build = ctx_module.build_initial_conversation

def _biased_build(*, baseline_eval, best_eval):
    convo = _original_build(baseline_eval=baseline_eval, best_eval=best_eval)
    convo.messages[-1]["content"] += f"\n\nADDITIONAL HEURISTIC GUIDANCE: {heuristic_bias}"
    return convo

gw_module.build_initial_conversation = _biased_build

reset_bot_to_baseline()
config = replace(STAGE_LOOP_CONFIG, iterations=5, mock_minimax=use_mock())
print(f"Running 5 iterations with heuristic bias:")
print(f"  {heuristic_bias!r}")
print()
summary_biased = run_loop(config)

# Restore the unbiased build so re-running other cells is unaffected
gw_module.build_initial_conversation = _original_build

# Compare
print()
print(f"=== TRAJECTORY COMPARISON ===")
unbiased_best = summary["best_eval"].get("estimated_elo") if "summary" in dir() else None
if unbiased_best is not None:
    print(f"  §4 unbiased run:  best Elo = {unbiased_best:.1f}")
print(f"  Biased run:       best Elo = {summary_biased['best_eval'].get('estimated_elo'):.1f}")
print(f"                    accepted = {summary_biased['accepted']} / {summary_biased['accepted'] + summary_biased['rejected']}")
print()
print(f"In LIVE mode, the bias actually changes what the agent proposes — different")
print(f"biases produce different Elo trajectories. Try several biases (e.g. 'aggressive")
print(f"king attack', 'positional/quiet play') and watch the curves diverge.")
print()
print(f"In MOCK mode, the agent's patches are scripted so the trajectory matches §4")
print(f"regardless of bias — the bias text is in the prompt but doesn't change behavior.")

## 7 · Where this leads next

The agent you ran in this notebook runs *in-process*: the four layers are Python modules in your kernel, the tools are local function calls. The same architecture is designed to deploy behind a service boundary — the OpenClaw Gateway runtime. The migration is mechanical because the seams are already there; see [`docs/openclaw_mapping.md`](./docs/openclaw_mapping.md) for the file-by-file mapping.

The deeper point is the one worth taking with you: **architectural seams are what make redeployment a deployment step instead of a rewrite.** Tools, Context, ReAct, and Gateway as separable layers is the property; in-process today and behind-a-service-boundary tomorrow is the consequence.

## 8 · Take this home

This notebook is the start, not the end. The pattern you used here transfers cleanly to other domains: anywhere you have an automatic metric, a bounded edit surface, and a clean rollback, you have the ingredients for an autoresearch loop.

- **The repository:** [`nickita-khylkouski/autoresearch-brief-challenge`](https://github.com/nickita-khylkouski/autoresearch-brief-challenge). Clone it for a real working environment.
- **The OpenClaw migration guide:** `docs/openclaw_mapping.md` in the repository.
- **The conceptual companion deck:** `Auto Research.pptx` — Karpathy's origin of this pattern, the MiniMax M2.7 case study at frontier scale, Ralph loops as a contrasting agent pattern, and the broader guardrails discussion (metric gaming, overfitting, hidden regressions, runaway loops).

### Save your run

Colab's filesystem is ephemeral. The next cell zips the trace, patches, and Elo curve you produced into a downloadable archive so you can keep them.

In [ ]:
"""Hand-off artifact: download your run."""
import shutil
import sys
from pathlib import Path

run_dir = Path(summary["run_dir"])
archive_base = Path.cwd() / f"workshop_run_{summary['run_id']}"
archive_path = shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=str(run_dir.parent),
    base_dir=run_dir.name,
)
print(f"Archive: {archive_path}")

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(archive_path)
else:
    print("(Not in Colab — archive saved to the path above; no browser download triggered.)")

---

### If your loop did not complete

Run the cell below only if the cells above failed for some reason. It replays a captured successful run from `artifacts/demo_replay/` so you can still inspect a working trace and curve.

In [ ]:
import sys
!{sys.executable} -m autoresearch_chess.replay --run artifacts/demo_replay